# Compare Dafne - to ground truth (extended metrics)

Note: this notebook was developed with assistance from Generative AI. The end cross schecks the results against a hand-coded earlier version

In [1]:
# libraries
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
!pwd
# above should be about ~something~/~someProjects~/dissector/eval_notebooks/myosegmenTUM

In [2]:
# dilation distance used for boundary_iou_3d
BOUNDARY_DISTANCE = 1

## Right gracilis

In [3]:
# unpack one npz to see muscle keys and find right gracilis
sample_npz = next(f for f in os.listdir("..") if f.endswith(".npz"))
data = np.load(os.path.join("..", sample_npz))

print(f"File: {sample_npz}\n")
for i, key in enumerate(data.files):
    marker = "  <-- RIGHT GRACILIS" if "gracilis" in key.lower() and "_r" in key.lower() else ""
    print(f"  [{i}] {key}{marker}")

File: HV001_1_FATFRACTION_stack1_dafne_thigh.npz

  [0] Vastus Lateralis_R
  [1] Vastus Lateralis_L
  [2] Vastus Medialis_R
  [3] Vastus Medialis_L
  [4] Vastus Intermedius_R
  [5] Vastus Intermedius_L
  [6] Rectus Femoris_R
  [7] Rectus Femoris_L
  [8] Sartorius_R
  [9] Sartorius_L
  [10] Gracilis_R  <-- RIGHT GRACILIS
  [11] Gracilis_L
  [12] Adductor Magnus_R
  [13] Adductor Magnus_L
  [14] Semimembranosus_R
  [15] Semimembranosus_L
  [16] Semitendinosus_R
  [17] Semitendinosus_L
  [18] Biceps Femoris Long_R
  [19] Biceps Femoris Long_L
  [20] Biceps Femoris Short_R
  [21] Biceps Femoris Short_L
  [22] Adductor Longus_R
  [23] Adductor Longus_L


In [ ]:
!pwd

In [ ]:
#find right gracilis

In [ ]:
results_r_gracilis = []
gt_base = os.path.join("..","..", "myosegmenTUM")

for file in os.listdir(os.path.join("..")):
    if not file.endswith(".npz"):
        continue
    print(file)

    # parse subject and stack number from dafne filename
    # e.g. HV001_1_FATFRACTION_stack1_dafne_thigh.npz
    subject = file.split("_FATFRACTION")[0]          # HV001_1 or P001_1
    stack_match = re.search(r"stack(\d+)", file)
    if not stack_match:
        print("  could not parse stack number, skipping")
        continue
    stack_num = stack_match.group(1)

    gt_name = os.path.join(gt_base, subject, "SegmentationMasks", f"combined_gt_stack{stack_num}.mha")
    print(gt_name)

    # load dafne prediction from npz — Gracilis_R is key [10]
    npz_data = np.load(os.path.join("..", file))
    pred_arr = npz_data["Gracilis_R"].astype(float)

    # load ground truth (label 5 = right gracilis)
    gt_image = sitk.ReadImage(gt_name)
    gt = sitk.Cast(gt_image == 5, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    # wrap pred as sitk image with same metadata for overlap filters
    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    right_grac_dice_lower = dice_filter.GetDiceCoefficient()
    right_JaccardCoeffi = dice_filter.GetJaccardCoefficient()
    right_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    right_FalseNegative = dice_filter.GetFalseNegativeError()
    right_FalsePositive = dice_filter.GetFalsePositiveError()

    hd_filter = sitk.HausdorffDistanceImageFilter()
    hd_filter.Execute(gt, pred)
    right_grac_hd_lower = hd_filter.GetHausdorffDistance()

    bce = binary_cross_entropy(gt_arr, pred_arr)
    biou = boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr)
    isd_pred = inter_slice_dice(pred_arr)
    isd_gt   = inter_slice_dice(gt_arr)

    results_r_gracilis.append({
        "image": gt_name,
        "pred_label": file,
        "R_gracilis_lower_dice:": right_grac_dice_lower,
        "R_gracilis_lower_Hausdorff:": right_grac_hd_lower,
        "R_gracilis_jaccard": right_JaccardCoeffi,
        "R_gracilis_volume_similarity": right_VolumeSimilar,
        "R_gracilis_falseNegative": right_FalseNegative,
        "R_gracilis_falsePostivie": right_FalsePositive,
        "R_gracilis_binary_cross_entropy": bce,
        "R_gracilis_boundary_iou_3d": biou,
        "R_gracilis_inter_slice_dice_pred": isd_pred,
        "R_gracilis_inter_slice_dice_gt": isd_gt,
    })

df_r_gracilis = pd.DataFrame(results_r_gracilis)
df_r_gracilis.to_csv("../results/df_r_gracilis_dafne.csv")
df_r_gracilis

## Left gracilis

In [ ]:
results_l_gracilis = []
gt_base = os.path.join("..","..", "myosegmenTUM")

for file in os.listdir(os.path.join("..")):
    if not file.endswith(".npz"):
        continue
    print(file)

    # parse subject and stack number from dafne filename
    # e.g. HV001_1_FATFRACTION_stack1_dafne_thigh.npz
    subject = file.split("_FATFRACTION")[0]          # HV001_1 or P001_1
    stack_match = re.search(r"stack(\d+)", file)
    if not stack_match:
        print("  could not parse stack number, skipping")
        continue
    stack_num = stack_match.group(1)

    gt_name = os.path.join(gt_base, subject, "SegmentationMasks", f"combined_gt_stack{stack_num}.mha")
    print(gt_name)

    # load dafne prediction from npz — Gracilis_L is key [11]
    npz_data = np.load(os.path.join("..", file))
    pred_arr = npz_data["Gracilis_L"].astype(float)

    # load ground truth (label 1 = left gracilis)
    gt_image = sitk.ReadImage(gt_name)
    gt = sitk.Cast(gt_image == 1, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    # wrap pred as sitk image with same metadata for overlap filters
    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    right_grac_dice_lower = dice_filter.GetDiceCoefficient()
    right_JaccardCoeffi = dice_filter.GetJaccardCoefficient()
    right_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    right_FalseNegative = dice_filter.GetFalseNegativeError()
    right_FalsePositive = dice_filter.GetFalsePositiveError()

    hd_filter = sitk.HausdorffDistanceImageFilter()
    hd_filter.Execute(gt, pred)
    right_grac_hd_lower = hd_filter.GetHausdorffDistance()

    bce = binary_cross_entropy(gt_arr, pred_arr)
    biou = boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr)
    isd_pred = inter_slice_dice(pred_arr)
    isd_gt   = inter_slice_dice(gt_arr)

    results_l_gracilis.append({
        "image": gt_name,
        "pred_label": file,
        "L_gracilis_lower_dice:": right_grac_dice_lower,
        "L_gracilis_lower_Hausdorff:": right_grac_hd_lower,
        "L_gracilis_jaccard": right_JaccardCoeffi,
        "L_gracilis_volume_similarity": right_VolumeSimilar,
        "L_gracilis_falseNegative": right_FalseNegative,
        "L_gracilis_falsePostivie": right_FalsePositive,
        "L_gracilis_binary_cross_entropy": bce,
        "L_gracilis_boundary_iou_3d": biou,
        "L_gracilis_inter_slice_dice_pred": isd_pred,
        "L_gracilis_inter_slice_dice_gt": isd_gt,
    }L

df_l_gracilis = pd.DataFrame(results_l_gracilis)
df_l_gracilis.to_csv("../results/df_l_gracilis_dafne.csv")
df_l_gracilis

## Right sartorius

In [5]:
results_r_sart = []
gt_base = os.path.join("..","..", "myosegmenTUM")

for file in os.listdir(os.path.join("..")):
    if not file.endswith(".npz"):
        continue
    print(file)

    # parse subject and stack number from dafne filename
    # e.g. HV001_1_FATFRACTION_stack1_dafne_thigh.npz
    subject = file.split("_FATFRACTION")[0]          # HV001_1 or P001_1
    stack_match = re.search(r"stack(\d+)", file)
    if not stack_match:
        print("  could not parse stack number, skipping")
        continue
    stack_num = stack_match.group(1)

    gt_name = os.path.join(gt_base, subject, "SegmentationMasks", f"combined_gt_stack{stack_num}.mha")
    print(gt_name)

    # load dafne prediction from npz — Sartorius_R is key [8]
    npz_data = np.load(os.path.join("..", file))
    pred_arr = npz_data["Sartorius_R"].astype(float)

    # load ground truth (label 8 = right sart)
    gt_image = sitk.ReadImage(gt_name)
    gt = sitk.Cast(gt_image == 8, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    # wrap pred as sitk image with same metadata for overlap filters
    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    right_sart_dice_lower = dice_filter.GetDiceCoefficient()
    right_JaccardCoeffi = dice_filter.GetJaccardCoefficient()
    right_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    right_FalseNegative = dice_filter.GetFalseNegativeError()
    right_FalsePositive = dice_filter.GetFalsePositiveError()

    hd_filter = sitk.HausdorffDistanceImageFilter()
    hd_filter.Execute(gt, pred)
    right_sart_hd_lower = hd_filter.GetHausdorffDistance()

    bce = binary_cross_entropy(gt_arr, pred_arr)
    biou = boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr)
    isd_pred = inter_slice_dice(pred_arr)
    isd_gt   = inter_slice_dice(gt_arr)

    results_r_sart.append({
        "image": gt_name,
        "pred_label": file,
        "R_sart_lower_dice:": right_sart_dice_lower,
        "R_sart_lower_Hausdorff:": right_sart_hd_lower,
        "R_sart_jaccard": right_JaccardCoeffi,
        "R_sart_volume_similarity": right_VolumeSimilar,
        "R_sart_falseNegative": right_FalseNegative,
        "R_sart_falsePostivie": right_FalsePositive,
        "R_sart_binary_cross_entropy": bce,
        "R_sart_boundary_iou_3d": biou,
        "R_sart_inter_slice_dice_pred": isd_pred,
        "R_sart_inter_slice_dice_gt": isd_gt,
    })

df_r_sart = pd.DataFrame(results_r_sart)
df_r_sart.to_csv("../results/df_r_sart_dafne.csv")
df_r_sart

HV001_1_FATFRACTION_stack1_dafne_thigh.npz
..\..\myosegmenTUM\HV001_1\SegmentationMasks\combined_gt_stack1.mha
HV001_1_FATFRACTION_stack2_dafne_thigh.npz
..\..\myosegmenTUM\HV001_1\SegmentationMasks\combined_gt_stack2.mha
HV001_2_FATFRACTION_stack1_dafne_thigh.npz
..\..\myosegmenTUM\HV001_2\SegmentationMasks\combined_gt_stack1.mha
HV001_2_FATFRACTION_stack2_dafne_thigh.npz
..\..\myosegmenTUM\HV001_2\SegmentationMasks\combined_gt_stack2.mha
HV001_3_FATFRACTION_stack1_dafne_thigh.npz
..\..\myosegmenTUM\HV001_3\SegmentationMasks\combined_gt_stack1.mha
HV001_3_FATFRACTION_stack2_dafne_thigh.npz
..\..\myosegmenTUM\HV001_3\SegmentationMasks\combined_gt_stack2.mha
HV002_1_FATFRACTION_stack1_dafne_thigh.npz
..\..\myosegmenTUM\HV002_1\SegmentationMasks\combined_gt_stack1.mha
HV002_1_FATFRACTION_stack2_dafne_thigh.npz
..\..\myosegmenTUM\HV002_1\SegmentationMasks\combined_gt_stack2.mha
HV002_2_FATFRACTION_stack1_dafne_thigh.npz
..\..\myosegmenTUM\HV002_2\SegmentationMasks\combined_gt_stack1.mha
H

,image,pred_label,R_sart_lower_dice:,R_sart_lower_Hausdorff:,R_sart_jaccard,R_sart_volume_similarity,R_sart_falseNegative,R_sart_falsePostivie,R_sart_binary_cross_entropy,R_sart_boundary_iou_3d,R_sart_inter_slice_dice_pred,R_sart_inter_slice_dice_gt
0,..\..\myosegmenTUM\HV001_1\SegmentationMasks\c...,HV001_1_FATFRACTION_stack1_dafne_thigh.npz,0.000000,240.220732,0.000000,0.771415,1.000000,0.000875,0.020347,0.000000,0.269198,0.815539
1,..\..\myosegmenTUM\HV001_1\SegmentationMasks\c...,HV001_1_FATFRACTION_stack2_dafne_thigh.npz,0.001136,184.304097,0.000568,1.459546,0.995795,0.000896,0.016684,0.004625,0.184092,0.787003
2,..\..\myosegmenTUM\HV001_2\SegmentationMasks\c...,HV001_2_FATFRACTION_stack1_dafne_thigh.npz,0.000000,236.433923,0.000000,0.613664,1.000000,0.000962,0.023718,0.000000,0.352351,0.832135
3,..\..\myosegmenTUM\HV001_2\SegmentationMasks\c...,HV001_2_FATFRACTION_stack2_dafne_thigh.npz,0.001974,180.502078,0.000988,1.743768,0.984595,0.001069,0.018393,0.007795,0.122834,0.844545
4,..\..\myosegmenTUM\HV001_3\SegmentationMasks\c...,HV001_3_FATFRACTION_stack1_dafne_thigh.npz,0.000000,230.139088,0.000000,0.808717,1.000000,0.001076,0.024694,0.000000,0.299994,0.866700
5,..\..\myosegmenTUM\HV001_3\SegmentationMasks\c...,HV001_3_FATFRACTION_stack2_dafne_thigh.npz,0.000000,186.796681,0.000000,1.687392,1.000000,0.001121,0.019598,0.004137,0.202757,0.851053
6,..\..\myosegmenTUM\HV002_1\SegmentationMasks\c...,HV002_1_FATFRACTION_stack1_dafne_thigh.npz,0.000000,225.588120,0.000000,1.068018,1.000000,0.001404,0.029483,0.000373,0.306179,0.872423
7,..\..\myosegmenTUM\HV002_1\SegmentationMasks\c...,HV002_1_FATFRACTION_stack2_dafne_thigh.npz,0.001010,171.735261,0.000505,1.642988,0.994343,0.001150,0.020338,0.012051,0.141783,0.872628
8,..\..\myosegmenTUM\HV002_2\SegmentationMasks\c...,HV002_2_FATFRACTION_stack1_dafne_thigh.npz,0.000000,234.855275,0.000000,0.897010,1.000000,0.001160,0.025800,0.000000,0.302764,0.854515
9,..\..\myosegmenTUM\HV002_2\SegmentationMasks\c...,HV002_2_FATFRACTION_stack2_dafne_thigh.npz,0.002053,165.704556,0.001028,1.647439,0.988352,0.001175,0.020753,0.010550,0.226193,0.872827


In [ ]:
# results_r_sart = []
# gt_base = os.path.join("..", "myosegmenTUM")
# for file in os.listdir(os.path.join("..", "MuscleMap_segs")):
#     print(file)
#     if file[0] == 'H':
#         gt_name = os.path.join(gt_base, file[:7], "SegmentationMasks", "combined_gt_stack" + file[-13] + ".mha")
#     else:
#         gt_name = os.path.join(gt_base, file[:6], "SegmentationMasks", "combined_gt_stack" + file[-13] + ".mha")
#     print(gt_name)

#     segment_image = sitk.ReadImage(os.path.join("..", "MuscleMap_segs", file))
#     gt_image = sitk.ReadImage(gt_name)
#     segment_image.CopyInformation(gt_image)

#     gt = sitk.Cast(gt_image == 8, sitk.sitkUInt8)
#     pred = sitk.Cast(segment_image == 7142, sitk.sitkUInt8)

#     gt_arr = sitk.GetArrayFromImage(gt).astype(float)
#     pred_arr = sitk.GetArrayFromImage(pred).astype(float)

#     dice_filter = sitk.LabelOverlapMeasuresImageFilter()
#     dice_filter.Execute(gt, pred)
#     right_sart_dice_lower = dice_filter.GetDiceCoefficient()
#     right_JaccardCoeffi = dice_filter.GetJaccardCoefficient()
#     right_VolumeSimilar = dice_filter.GetVolumeSimilarity()
#     right_FalseNegative = dice_filter.GetFalseNegativeError()
#     right_FalsePositive = dice_filter.GetFalsePositiveError()

#     hd_filter = sitk.HausdorffDistanceImageFilter()
#     hd_filter.Execute(gt, pred)
#     right_sart_hd_lower = hd_filter.GetHausdorffDistance()

#     bce = binary_cross_entropy(gt_arr, pred_arr)
#     biou = boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr)

#     results_r_sart.append({
#         "image": gt_name,
#         "pred_label": file,
#         "R_sartorius_lower_dice:": right_sart_dice_lower,
#         "R_sartorius_lower_Hausdorff:": right_sart_hd_lower,
#         "R_sartorius_jaccard": right_JaccardCoeffi,
#         "R_sartorius_volume_similarity": right_VolumeSimilar,
#         "R_sartorius_falseNegative": right_FalseNegative,
#         "R_sartorius_falsePostivie": right_FalsePositive,
#         "R_sartorius_binary_cross_entropy": bce,
#         "R_sartorius_boundary_iou_3d": biou,
#     })

# df_r_sart = pd.DataFrame(results_r_sart)
# df_r_sart.to_csv('../MuscleMap_results_WB/df_r_sart_extended.csv')
# df_r_sart

## Left sartorius

In [ ]:
results_l_sart = []
gt_base = os.path.join("..", "myosegmenTUM")
for file in os.listdir(os.path.join("..", "MuscleMap_segs")):
    print(file)
    if file[0] == 'H':
        gt_name = os.path.join(gt_base, file[:7], "SegmentationMasks", "combined_gt_stack" + file[-13] + ".mha")
    else:
        gt_name = os.path.join(gt_base, file[:6], "SegmentationMasks", "combined_gt_stack" + file[-13] + ".mha")
    print(gt_name)

    segment_image = sitk.ReadImage(os.path.join("..", "MuscleMap_segs", file))
    gt_image = sitk.ReadImage(gt_name)
    segment_image.CopyInformation(gt_image)

    gt = sitk.Cast(gt_image == 4, sitk.sitkUInt8)
    pred = sitk.Cast(segment_image == 7141, sitk.sitkUInt8)

    gt_arr = sitk.GetArrayFromImage(gt).astype(float)
    pred_arr = sitk.GetArrayFromImage(pred).astype(float)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    left_sart_dice_lower = dice_filter.GetDiceCoefficient()
    left_JaccardCoeffi = dice_filter.GetJaccardCoefficient()
    left_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    left_FalseNegative = dice_filter.GetFalseNegativeError()
    left_FalsePositive = dice_filter.GetFalsePositiveError()

    hd_filter = sitk.HausdorffDistanceImageFilter()
    hd_filter.Execute(gt, pred)
    left_sart_hd_lower = hd_filter.GetHausdorffDistance()

    bce = binary_cross_entropy(gt_arr, pred_arr)
    biou = boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr)

    results_l_sart.append({
        "image": gt_name,
        "pred_label": file,
        "L_sartorius_lower_dice:": left_sart_dice_lower,
        "L_sartorius_lower_Hausdorff:": left_sart_hd_lower,
        "L_sartorius_jaccard": left_JaccardCoeffi,
        "L_sartorius_volume_similarity": left_VolumeSimilar,
        "L_sartorius_falseNegative": left_FalseNegative,
        "L_sartorius_falsePostivie": left_FalsePositive,
        "L_sartorius_binary_cross_entropy": bce,
        "L_sartorius_boundary_iou_3d": biou,
    })

df_l_sart = pd.DataFrame(results_l_sart)
df_l_sart.to_csv('../MuscleMap_results_WB/df_l_sart_extended.csv')
df_l_sart